# Happy New Year 2026 - Controlled Particle Swarm Animation

This notebook creates a New Year animation using PD-controlled particles that form text,
similar to drone light shows. The particles use feedback control to converge from random
positions into the text "HAPPY NEW YEAR 2026", with firecracker effects for celebration.

In [ ]:
using DifferentialEquations, GLMakie, Random, LinearAlgebra

Random.seed!(333)

In [ ]:
# Combined dynamics: PD-controlled text particles + ballistic firecracker particles
function combined_dynamics!(du, u, p, t)
    Kp, Kd, c, g = p
    
    # Text particles (controlled with PD)
    for i in 1:n_particles
        idx = 4*(i-1)
        x = u[idx+1]
        y = u[idx+2]
        vx = u[idx+3]
        vy = u[idx+4]
        
        tx, ty = targets[i]
        
        # PD control: acceleration = Kp*(error) - Kd*velocity - damping*velocity
        ax = Kp*(tx - x) - Kd*vx - c*vx
        ay = Kp*(ty - y) - Kd*vy - c*vy
        
        du[idx+1] = vx
        du[idx+2] = vy
        du[idx+3] = ax
        du[idx+4] = ay
    end
    
    # Firecracker particles (ballistic with gravity)
    for i in 1:n_crackers
        idx = 4*(n_particles + i - 1)
        vx = u[idx+3]
        vy = u[idx+4]
        
        du[idx+1] = vx
        du[idx+2] = vy
        du[idx+3] = -0.5 * vx
        du[idx+4] = -g - 0.5 * vy
    end
end

In [ ]:
# Letter patterns - coordinate definitions for each character
function get_letter_improved(letter, x_offset, y_offset, scale=4.0)
    patterns = Dict(
        'H' => [(0,0), (0,1), (0,2), (0,3), (0,4), (1,2), (2,0), (2,1), (2,2), (2,3), (2,4)],
        'A' => [(0,0), (0.3,0.7), (0.6,1.4), (0.8,2.1), (0.9,2.8), (1,3.5),
                (2,0), (1.7,0.7), (1.4,1.4), (1.2,2.1), (1.1,2.8), (1,3.5),
                (0.7,1.5), (1,1.5), (1.3,1.5)],
        'P' => [(0,0), (0,1), (0,2), (0,3), (0,4), (0.5,4), (1,4), (1.5,3.7), (1.5,3.3), (1.2,2.8), (0.7,2.5), (0.3,2.5)],
        'Y' => [(0,4), (0.4,3.5), (0.7,3), (1,2.5), (2,4), (1.6,3.5), (1.3,3), (1,2.5),
                (1,2), (1,1.5), (1,1), (1,0.5), (1,0)],
        'N' => [(0,0), (0,1), (0,2), (0,3), (0,4), (0.5,3.5), (1,3), (1.5,2), (2,0), (2,1), (2,2), (2,3), (2,4)],
        'E' => [(0,0), (0,1), (0,2), (0,3), (0,4), (0.7,0), (1.4,0), (0.7,2), (1.2,2), (0.7,4), (1.4,4)],
        'W' => [(0,0), (0,1), (0,2), (0,3), (0,4), (0.5,0.5), (0.7,1), (0.8,1.5),
                (1,0), (1,0.5), (1,1), (1.2,1.5), (1.3,1), (1.5,0.5),
                (2,0), (2,1), (2,2), (2,3), (2,4)],
        'R' => [(0,0), (0,1), (0,2), (0,3), (0,4), (0.5,4), (1,4), (1.5,3.7), (1.5,3.3), (1.2,2.8), (0.5,2.5), (1,2), (1.5,1.2), (2,0.3)],
        '2' => [(0,4), (0.6,4), (1.2,4), (1.8,4), (2,3.7), (2,3.3), (1.6,2.8), (1.2,2.4), (0.7,2), (0.3,1.5), (0,1), (0,0.5), (0,0), (0.6,0), (1.2,0), (1.8,0)],
        '0' => [(1,0), (0.4,0.5), (0,1.2), (0,2), (0,2.8), (0.4,3.5), (1,4), (1.6,3.5), (2,2.8), (2,2), (2,1.2), (1.6,0.5)],
        '6' => [(1,4), (0.5,3.6), (0.2,3.1), (0,2.5), (0,1.8), (0,1.1), (0.3,0.5), (0.8,0.1), (1.4,0), (1.9,0.4), (2.1,1), (2,1.6), (1.6,2), (1,2.1), (0.5,1.9)]
    )
    
    if !haskey(patterns, letter)
        return []
    end
    
    return [(x_offset + x * scale, y_offset + y * scale) for (x, y) in patterns[letter]]
end

In [ ]:
# Generate target positions for "HAPPY NEW YEAR 2026"
text = "HAPPY NEW YEAR 2026"
targets = []

x_start = 80.0
y_base = 35.0
letter_spacing = 10.0
word_spacing = 14.0
x_pos = x_start

for char in text
    if char == ' '
        x_pos += word_spacing
        continue
    end
    points = get_letter_improved(char, x_pos, y_base, 2.8)
    append!(targets, points)
    x_pos += letter_spacing
end

n_particles = length(targets)
println("Generated $(n_particles) text particles")

In [ ]:
# Create firecracker burst particles
function create_firecracker_burst(x_center, y_center, burst_time, n=30)
    particles = []
    for i in 1:n
        angle = 2π * i / n + randn() * 0.3
        speed = 15.0 + randn() * 5.0
        vx = speed * cos(angle)
        vy = speed * sin(angle)
        x0 = x_center + randn() * 2.0
        y0 = y_center + randn() * 2.0
        push!(particles, (x0, y0, vx, vy, burst_time))
    end
    return particles
end

firecracker_bursts = []
append!(firecracker_bursts, create_firecracker_burst(100.0, 55.0, 2.0, 20))
append!(firecracker_bursts, create_firecracker_burst(140.0, 56.0, 2.5, 20))
append!(firecracker_bursts, create_firecracker_burst(180.0, 55.0, 3.0, 20))
append!(firecracker_bursts, create_firecracker_burst(70.0, 40.0, 3.5, 20))
append!(firecracker_bursts, create_firecracker_burst(210.0, 40.0, 4.0, 20))

n_crackers = length(firecracker_bursts)
println("Created $(n_crackers) firecracker particles")

In [ ]:
# Initial conditions: text particles scattered, firecrackers at burst positions
u0 = Float64[]

for i in 1:n_particles
    push!(u0, 140.0 + randn()*40.0)  # x
    push!(u0, 40.0 + randn()*15.0)   # y
    push!(u0, 0.0)                    # vx
    push!(u0, 0.0)                    # vy
end

for (x0, y0, vx0, vy0, _) in firecracker_bursts
    push!(u0, x0)
    push!(u0, y0)
    push!(u0, vx0)
    push!(u0, vy0)
end

In [ ]:
# Solve ODE system
params = (22.0, 11.0, 0.6, 9.81)  # Kp, Kd, damping, gravity
tspan = (0.0, 6.0)
prob = ODEProblem(combined_dynamics!, u0, tspan, params)

println("Solving ODE system...")
sol = solve(prob, Tsit5(), saveat=0.04)
println("✓ Solved! $(length(sol.t)) time points")

In [ ]:
# Extract particle data for each frame
function get_all_particles(t_idx)
    text_points = Point2f[]
    text_colors = Float32[]
    cracker_points = Point2f[]
    cracker_colors = Float32[]
    
    t_current = sol.t[t_idx]
    
    # Text particles
    for i in 1:n_particles
        idx = 4*(i-1)
        x = sol[idx+1, t_idx]
        y = sol[idx+2, t_idx]
        tx, ty = targets[i]
        dist = sqrt((x - tx)^2 + (y - ty)^2)
        color_val = Float32(max(0.0, 1.0 - dist/18.0))
        push!(text_points, Point2f(x, y))
        push!(text_colors, color_val)
    end
    
    # Firecracker particles
    for i in 1:n_crackers
        _, _, _, _, burst_time = firecracker_bursts[i]
        if t_current >= burst_time
            idx = 4*(n_particles + i - 1)
            x = sol[idx+1, t_idx]
            y = sol[idx+2, t_idx]
            if y > 10
                age = t_current - burst_time
                brightness = Float32(max(0.0, 1.0 - age/1.5))
                push!(cracker_points, Point2f(x, y))
                push!(cracker_colors, brightness)
            end
        end
    end
    
    return text_points, text_colors, cracker_points, cracker_colors
end

In [ ]:
# Create and record animation
fig = Figure(size=(1200, 600))
ax = Axis(fig[1,1],
    aspect=DataAspect(),
    limits=(70..285, 27..58),
    backgroundcolor=:black,
    leftspinevisible=false,
    rightspinevisible=false,
    topspinevisible=false,
    bottomspinevisible=false,
    leftspinecolor=:black,
    rightspinecolor=:black,
    topspinecolor=:black,
    bottomspinecolor=:black
)

hidedecorations!(ax)

tp0, tc0, cp0, cc0 = get_all_particles(1)

text_scatter = GLMakie.scatter!(ax, tp0, 
    color=tc0, 
    markersize=15, 
    colormap=:hot,
    colorrange=(0.0f0, 1.0f0)
)

cracker_scatter = GLMakie.scatter!(ax, cp0,
    color=cc0,
    markersize=9,
    colormap=:thermal,
    colorrange=(0.0f0, 1.0f0)
)

println("Recording animation...")

record(fig, "Happy_NewYear_2026.mp4", eachindex(sol.t); framerate=25) do i
    tp, tc, cp, cc = get_all_particles(i)
    text_scatter[1][] = tp
    text_scatter.color = tc
    cracker_scatter[1][] = cp
    cracker_scatter.color = cc
end

println("✓ Animation created: Happy_NewYear_2026.mp4")